In [1]:
print("Hello")

Hello


In [2]:
import pandas as pd;
import openai

In [3]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance,VectorParams,PointStruct

Read And Sample dataset with Amazon inventory data

In [4]:
df_items=pd.read_json("C:\\Users\\jaysi\\Desktop\\Desktop\\Ai-engineering\\data\\meta_Electronics_2022_23_with_categeory_rating_100_sample_2000.jsonl",lines=True)


In [5]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,[],4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,[],NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[],169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...",[],24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN


In [6]:
list(df_items['features'].items())[0]

(0,
 ['【Fast Charging Cord】These USB C cables provide up to a 3A charging current to greatly shorten the charging time, meets QC2.0 /3.0 fast charging protocol,Incredibly charge your phone from 0 to 80% in 50 minute. 480Mbps (40-60M/s) ultra fast data transmission, which leads to a faster data sync.(Note:Cables support fast charging,but require a USB-A QC3.0/QC2.0/AFC charger)',
  '【Universal Compatibility】The USB C Charger Cable is compatible with Samsung Galaxy S20 / S10 / S9 / S8+ / S8 / A02s / A03s,A12 A20 A21 A22 A23 A31 A32 A33 A41 A42 A50 A52 A52s 5G A71 A72 A73,M11 M21 M31 M51,M12 M22 M32 M52,iPad Pro 2018 / 2020,Sony Xperia XZ/X Compact/L1 / XZs / XA1 / X Premium, HTC 10 LG G5 G6,OnePlus 5T / 6T, Lumia 950 / 950XL,Huawei P9 P9 Plus P10 P10 Plus Honor Mate 9 Mate 9 pro Mate 10 pro Mate 10 lite and more',
  '【Premium Workmanship】Unique increased friction design allows you to easily unplug the cable from your charger,combine 250d bulletproof fiber core to build a cable so durable

In [7]:
list(df_items['images'].items())[0]

(0,
 [{'thumb': 'https://m.media-amazon.com/images/I/51G07yWoOBL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51G07yWoOBL.jpg',
   'variant': 'MAIN',
   'hi_res': 'https://m.media-amazon.com/images/I/611AVJMH+JL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41c+40oKkKL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41c+40oKkKL.jpg',
   'variant': 'PT01',
   'hi_res': 'https://m.media-amazon.com/images/I/61ihhPW7VCL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/51y1YnwiUZL._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/51y1YnwiUZL.jpg',
   'variant': 'PT02',
   'hi_res': 'https://m.media-amazon.com/images/I/61UkcVETKcL._SL1200_.jpg'},
  {'thumb': 'https://m.media-amazon.com/images/I/41Nvr++q39L._SX38_SY50_CR,0,0,38,50_.jpg',
   'large': 'https://m.media-amazon.com/images/I/41Nvr++q39L.jpg',
   'variant': 'PT03',
   'hi_res': 'https://m.media-amazon.c

# Preprocess Title And Feature

In [8]:
def preprocessed_decription(row):
    return f"{row['title']}{''.join(row['features'])}"

In [9]:
def extrect_first_large_image(row):
    return row['images'][0].get("large","")

In [10]:
df_items['description']=df_items.apply(preprocessed_decription,axis=1)
df_items['image']=df_items.apply(extrect_first_large_image,axis=1)

In [11]:
df_items.head()

,main_category,title,average_rating,rating_number,features,description,price,images,videos,store,categories,details,parent_asin,bought_together,subtitle,author,image
0,Industrial & Scientific,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",4.4,119,[【Fast Charging Cord】These USB C cables provid...,"RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB T...",NaN,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Type-C Charger Cable ', 'url': 'ht...",RAVODOI,"[Electronics, Computers & Accessories, Compute...","{'Brand': 'RAVODOI', 'Connector Type': 'USB Ty...",B09R4Y2HKY,NaN,NaN,NaN,https://m.media-amazon.com/images/I/51G07yWoOB...
1,All Electronics,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.5,352,[🔹(Light & compact) Easy to carry and light we...,"SNESH-2 Pack USB-C Female to USB Male Adapter,...",4.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'USB Male & Female Adapter', 'url':...",SNESH,"[Electronics, Computers & Accessories, Compute...",{'Package Dimensions': '3.54 x 2.4 x 0.35 inch...,B09JV5FM2S,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41bOA5-ogW...
2,All Electronics,USB C Docking Station Dual Monitor for MacBook...,3.9,1193,[【18-in-1Docking Station】With USB C Docking St...,USB C Docking Station Dual Monitor for MacBook...,NaN,[{'thumb': 'https://m.media-amazon.com/images/...,[],ZMUIPNG,"[Electronics, Computers & Accessories, Laptop ...","{'Product Dimensions': '3.94""L x 1.18""W x 3.94...",B09SFN9NRX,NaN,NaN,NaN,https://m.media-amazon.com/images/I/416IzmVKiC...
3,Camera & Photo,[2023 Upgraded] Telescopes for Adults Astronom...,4.1,219,[🎁【2023 All New Experience】The newly upgraded ...,[2023 Upgraded] Telescopes for Adults Astronom...,169.99,[{'thumb': 'https://m.media-amazon.com/images/...,"[{'title': 'Good picture quality', 'url': 'htt...",HUTACT,"[Electronics, Camera & Photo, Binoculars & Sco...","{'Product Dimensions': '32.5""D x 5.5""W x 9.7""H...",B09TP3SZ7C,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41wO4J3TT0...
4,AMAZON FASHION,"Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",4.5,222,"[Leather,Mesh, Imported, Multi-pockets and Lar...","Laptop Bag 15.6 Inch, Laptop Briefcase Messeng...",24.95,[{'thumb': 'https://m.media-amazon.com/images/...,[],KPIQIU,"[Electronics, Computers & Accessories, Laptop ...",{'Product Dimensions': '16 x 2 x 12 inches; 1....,B0B5H7T7XZ,NaN,NaN,NaN,https://m.media-amazon.com/images/I/41mwlYqT5p...


In [12]:
list(df_items['description'].items())[0]

(0,
 "RAVODOI USB C Cable, [2Pack/3.3ft+6.6ft] USB Type C Fast Charging Cord - Nylon Braided USB C Charger Cable for Galaxy A20/A50/S10/S9/S8+/S8, iPad Pro 2018, Sony XZ, HTC 10, OnePlus 5T, Huawei P9 etc.【Fast Charging Cord】These USB C cables provide up to a 3A charging current to greatly shorten the charging time, meets QC2.0 /3.0 fast charging protocol,Incredibly charge your phone from 0 to 80% in 50 minute. 480Mbps (40-60M/s) ultra fast data transmission, which leads to a faster data sync.(Note:Cables support fast charging,but require a USB-A QC3.0/QC2.0/AFC charger)【Universal Compatibility】The USB C Charger Cable is compatible with Samsung Galaxy S20 / S10 / S9 / S8+ / S8 / A02s / A03s,A12 A20 A21 A22 A23 A31 A32 A33 A41 A42 A50 A52 A52s 5G A71 A72 A73,M11 M21 M31 M51,M12 M22 M32 M52,iPad Pro 2018 / 2020,Sony Xperia XZ/X Compact/L1 / XZs / XA1 / X Premium, HTC 10 LG G5 G6,OnePlus 5T / 6T, Lumia 950 / 950XL,Huawei P9 P9 Plus P10 P10 Plus Honor Mate 9 Mate 9 pro Mate 10 pro Mate 10 

In [13]:
list(df_items['image'].items())[0]

(0, 'https://m.media-amazon.com/images/I/51G07yWoOBL.jpg')

In [14]:
df_sample=df_items.sample(50,random_state=42)

In [15]:
data_to_embed=df_sample[["description","image","rating_number","price","average_rating","parent_asin"]].to_dict(orient="records")

In [31]:
length(data_to_embed)

NameError: name 'length' is not defined

## Embedding Function

In [27]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
client = OpenAI()

In [33]:
response=client.embeddings.create(
        model="text-embedding-3-small",
        input="random  text"
    )

In [39]:
len(response.data[0].embedding)

1536

In [29]:
def create_embeddings(text,model="text-embedding-3-small"):
    response=client.embeddings.create(
        model=model,
        input=text
    )
    return response.data[0].embedding
    

In [30]:
quadrant_client=QdrantClient(url="http://localhost:6333/")

In [ ]:
quadrant_client.create_collection(
    collection_name="Amazon_items_collection-00",
    vectors_config=VectorParams(size=1536,distance=Distance.COSINE)
)

In [45]:

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

In [46]:
points_struct = []

for i, data in enumerate(data_to_embed):
    embedding = get_embedding(data["description"])

    points_struct.append(
        PointStruct(
            id=i,
            vector=embedding,
            payload=data
        )
    )

In [48]:
points_struct

[PointStruct(id=0, vector=[-0.0003147125244140625, 0.028961181640625, -0.0189666748046875, -0.03216552734375, -0.0374755859375, -0.005435943603515625, -0.0011081695556640625, 0.0142059326171875, 0.0226287841796875, -0.017730712890625, 0.031402587890625, -0.0482177734375, -0.0278167724609375, 0.02972412109375, 0.01837158203125, -0.00643157958984375, -0.05303955078125, -0.026763916015625, -0.031646728515625, -0.0058746337890625, 0.006107330322265625, 0.051422119140625, -0.042327880859375, -0.007259368896484375, 0.0086822509765625, -0.002910614013671875, 0.03143310546875, 0.02392578125, 0.01322174072265625, -0.01024627685546875, 0.0187225341796875, 0.006591796875, 0.0015649795532226562, -0.07220458984375, -0.01384735107421875, -0.031463623046875, 0.0382080078125, -0.029815673828125, 0.0204925537109375, 0.006317138671875, 0.0185394287109375, 0.00848388671875, 0.00554656982421875, 0.00930023193359375, -0.01111602783203125, 0.0345458984375, -0.007293701171875, 0.03900146484375, -0.0191192626

Write embedded data to Qadrant

In [49]:
quadrant_client.upsert(
    collection_name="Amazon_items_collection-00",
    wait=True,
    points=points_struct
    
)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

simple Retriver fuction

In [66]:
def retrive_data(query, k=5):
    query_embedding = get_embedding(query)

    result = quadrant_client.query_points(
        collection_name="Amazon_items_collection-00",
        query=query_embedding,
        limit=k,
    )

    print("Inside function:", result)  # Debug

    return result

In [67]:
res = retrive_data("what kind of charging cords you offered", 5)

print("Outside function:", res)
print(type(res))

Inside function: points=[ScoredPoint(id=47, version=1, score=0.52936536, payload={'description': '5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB-C to LIGHTNING charging cord】UNIVERSAL COMPATIBILITY - Durable braided lightning cords are fully compatible with Apple iPhone 14 Pro Max, 14 Plus, 14 Pro, 14, 13 Pro Max, 13 Pro, 13, 13 mini, 12 Pro Max, 12 Pro, 12, 12 mini, SE, 11 Pro Max, 11 Pro, 11, XS Max, XS, XR, X, 8Plus, 8, 7 Plus, 7, SE, 6s Plus, 6s, 6Plus, 6; iPad 10.2 (20

In [71]:
result.points[0]

ScoredPoint(id=47, version=1, score=0.5293168, payload={'description': '5pack iPhone Charger Lightning Cables for Phone 13 13pro Max 12 Mini 11 Se 10 X Xr Xs 8 7 6 6s, Apple Certified Original Braided Cord Multi-Color Lightening Wire Fast Charging Cargador Chord-3 6 10 FtAPPLE MFI CERTIFIED - Made with original Apple MFI Chip, full support with iOS versions. High quality, pull-resistance, fast charging, and transmission.USB A TO LIGHTNING - USB-A to USB-Lightning port cable matches with USB A Wall charger. you can charge your iPhone, iPad, iPod. But not fit for android micro or type c port phones. 【Please make sure this is not a USB-C to LIGHTNING charging cord】UNIVERSAL COMPATIBILITY - Durable braided lightning cords are fully compatible with Apple iPhone 14 Pro Max, 14 Plus, 14 Pro, 14, 13 Pro Max, 13 Pro, 13, 13 mini, 12 Pro Max, 12 Pro, 12, 12 mini, SE, 11 Pro Max, 11 Pro, 11, XS Max, XS, XR, X, 8Plus, 8, 7 Plus, 7, SE, 6s Plus, 6s, 6Plus, 6; iPad 10.2 (2021), iPad 10.2 (2020), iPa